# Grounded radiology reporting — Gradio interface



## 1. Install and mount

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, random, hashlib, shutil, subprocess, io
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image, ImageDraw, ImageFont

OUT_ROOT = "/content/drive/MyDrive/MSC_VLM_GROUNDED"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

PNG_TAR, PNG_LOCAL = f"{OUT_ROOT}/pcgr_png.tar", "/content/pc_png"
if not os.path.exists(f"{PNG_LOCAL}/manifest.json"):
    if not os.path.isdir(PNG_LOCAL) or len(os.listdir(PNG_LOCAL)) < 100:
        subprocess.run(["tar", "-xf", PNG_TAR, "-C", "/content"])
    if not os.path.exists(f"{PNG_LOCAL}/manifest.json"):
        shutil.copy(f"{OUT_ROOT}/manifest.json", f"{PNG_LOCAL}/manifest.json")
pc_all = json.load(open(f"{PNG_LOCAL}/manifest.json"))
print(f"{len(pc_all)} studies available for examples")

Mounted at /content/drive
device: cuda
4555 studies available for examples


## 2. Definitions

In [ ]:
import re

class CFG:
    SEED = 42

    # vision
    IMG_SIZE = 518
    PATCH = 14
    RAW_GRID = 37                  # 518 / 14
    POOL_GRID = 19
    VISION_DIM = 768

    # coordinate vocabulary
    COORD_BINS = 32                # -> 64 tokens (32 x + 32 y)

    # llm
    LLM_ID = "Qwen/Qwen2.5-3B-Instruct"
    LORA_R = 32
    LORA_ALPHA = 64
    LORA_DROPOUT = 0.05
    LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

    # training
    EPOCHS = 1
    BATCH = 1
    ACCUM = 8
    LR = 2e-6
    LR_ADAPTER = 5e-5
    WARMUP_FRAC = 0.03
    MAX_LEN = 512
    GROUNDED_FRACTION = 1.0

    # decoding
    MAX_NEW_TOKENS = 220
    DO_SAMPLE = True
    TOP_P = 0.9
    TEMPERATURE = 0.7


def seed_everything(s=CFG.SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ==============================================================================
# PROMPTS
# ==============================================================================

SYSTEM = ("<|im_start|>system\nYou are an expert radiology assistant tasked "
          "with interpreting a chest X-ray study.<|im_end|>\n")

INSTR_FINDGEN = (
    "<|im_start|>user\nProvide a description of the findings in the radiology "
    "study. Each finding should be described as a self-contained plain-text "
    "sentence.<|im_end|>\n<|im_start|>assistant\n")

INSTR_GROUNDREP = (
    "<|im_start|>user\nProvide a description of the findings in the radiology "
    "study. Each finding should be described as a self-contained plain-text "
    "sentence. If the finding is groundable, locate it in the image with "
    "bounding boxes indicating all locations where it can be seen. Otherwise "
    "generate just the ungrounded finding.<|im_end|>\n<|im_start|>assistant\n")


# ==============================================================================
# COORDINATE VOCABULARY
# ==============================================================================

OBJ_OPEN, OBJ_CLOSE = "<obj>", "</obj>"
BOX_OPEN, BOX_CLOSE = "<box>", "</box>"


def coord_tokens(n=CFG.COORD_BINS):
    return ([f"<x{i}>" for i in range(n)] + [f"<y{i}>" for i in range(n)]
            + [OBJ_OPEN, OBJ_CLOSE, BOX_OPEN, BOX_CLOSE])


def add_coord_tokens(tokenizer, model, n=CFG.COORD_BINS):

    new = coord_tokens(n)
    added = tokenizer.add_tokens(new, special_tokens=True)
    if added == 0:
        return tokenizer, model
    model.resize_token_embeddings(len(tokenizer))
    with torch.no_grad():
        emb = model.get_input_embeddings().weight
        old = emb[:-added].float()
        mu, sd = old.mean(0, keepdim=True), old.std(0, keepdim=True)
        emb[-added:] = (mu + torch.randn(added, emb.shape[1],
                                         device=emb.device) * sd).to(emb.dtype)
        out = model.get_output_embeddings()
        if out is not None and out.weight.data_ptr() != emb.data_ptr():
            o = out.weight[:-added].float()
            om, os_ = o.mean(0, keepdim=True), o.std(0, keepdim=True)
            out.weight[-added:] = (om + torch.randn(added, o.shape[1],
                                                    device=o.device) * os_
                                   ).to(out.weight.dtype)
        chk = emb[-added:]
        spread = chk.float().std(0).mean().item()
    print(f"added {added} tokens | vocab {len(tokenizer)} | "
          f"new-row spread {spread:.5f} (must be > 0)")
    assert spread > 1e-4
    return tokenizer, model


def box_to_tokens(box_norm, n=CFG.COORD_BINS):

    x0, y0, x1, y1 = box_norm
    q = lambda v: int(np.clip(round(v * (n - 1)), 0, n - 1))
    return f"<x{q(x0)}><y{q(y0)}><x{q(x1)}><y{q(y1)}>"


BOX_RE = re.compile(r"<x(\d+)><y(\d+)><x(\d+)><y(\d+)>")


def parse_generation(text, n=CFG.COORD_BINS):

    out = []
    for chunk in re.findall(r"<obj>(.*?)</obj>", text, flags=re.S):
        boxes = [(int(a)/(n-1), int(b)/(n-1), int(c)/(n-1), int(d)/(n-1))
                 for a, b, c, d in BOX_RE.findall(chunk)]
        sent = re.sub(r"<box>.*?</box>", "", chunk, flags=re.S)
        sent = BOX_RE.sub("", sent).strip()
        if sent:
            out.append((sent, boxes))
    if not out and text.strip():
        out = [(text.strip(), [])]
    return out


# ==============================================================================
# GEOMETRY — image and boxes
# ==============================================================================

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)


def preprocess(pil_img, boxes_xyxy=None, size=CFG.IMG_SIZE):

    W, H = pil_img.size
    scale = size / min(W, H)
    nw, nh = int(round(W * scale)), int(round(H * scale))
    img = pil_img.convert("RGB").resize((nw, nh), Image.BILINEAR)
    ox, oy = max(0, (nw - size) // 2), max(0, (nh - size) // 2)
    img = img.crop((ox, oy, ox + size, oy + size))

    arr = np.asarray(img, np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    px = torch.from_numpy(arr).permute(2, 0, 1)

    if boxes_xyxy is None:
        return px, None, img

    out = []
    for b in boxes_xyxy:
        nb = (b[0]*scale - ox, b[1]*scale - oy, b[2]*scale - ox, b[3]*scale - oy)
        cl = (max(0., nb[0]), max(0., nb[1]),
              min(float(size), nb[2]), min(float(size), nb[3]))
        oa = max(1e-6, (nb[2]-nb[0]) * (nb[3]-nb[1]))
        sa = max(0., cl[2]-cl[0]) * max(0., cl[3]-cl[1])
        out.append(tuple(v / size for v in cl) if sa/oa >= 0.30 else None)
    return px, out, img




In [ ]:
# ==============================================================================
# TARGET FORMATTING
# ==============================================================================

def format_findgen(sentences):

    return " ".join(s.strip() for s in sentences if s.strip())


def format_groundrep(sentences, boxes_per_sentence):

    parts = []
    for sent, boxes in zip(sentences, boxes_per_sentence):
        s = sent.strip()
        if not s:
            continue
        valid = [b for b in boxes if b is not None]
        if valid:
            btxt = "".join(BOX_OPEN + box_to_tokens(b) + BOX_CLOSE for b in valid)
            parts.append(f"{OBJ_OPEN}{s}{btxt}{OBJ_CLOSE}")
        else:
            parts.append(f"{OBJ_OPEN}{s}{OBJ_CLOSE}")
    return "".join(parts)


# ==============================================================================
# ABSTENTION — three categories
# ==============================================================================

NEGATION_CUES = ["no ", "not ", "without", "free of", "clear of", "negative for",
                 "absence of", "absent", "unremarkable", "normal limits",
                 "within normal", "no evidence"]
NORMALITY_CUES = ["are clear", "is clear", "is normal", "are normal",
                  "unremarkable", "within normal limits", "no abnormalit",
                  "intact", "well-inflated", "well expanded"]
NON_LOCALISABLE_CUES = ["diffuse", "generalised", "generalized", "widespread",
                        "throughout", "global", "hyperinflat", "hypoinflat",
                        "low lung volumes", "emphysema", "copd", "osteopenia",
                        "deminerali", "scoliosis"]


def is_groundable(sentence):
    s = " " + sentence.lower().strip() + " "
    if any(c in s for c in NEGATION_CUES):        return False
    if any(c in s for c in NORMALITY_CUES):       return False
    if any(c in s for c in NON_LOCALISABLE_CUES): return False
    return True


# ==============================================================================
# SPLITS
# ==============================================================================

def study_key(text):

    return hashlib.md5(re.sub(r"\s+", " ", (text or "").strip().lower())
                       .encode()).hexdigest()


def group_split(keys, frac=(0.8, 0.1, 0.1), seed=CFG.SEED):
    uniq = sorted(set(keys)); random.Random(seed).shuffle(uniq)
    n, a, b = len(uniq), int(len(uniq)*frac[0]), int(len(uniq)*(frac[0]+frac[1]))
    assign = {k: ("train" if i < a else "val" if i < b else "test")
              for i, k in enumerate(uniq)}
    out = {"train": [], "val": [], "test": []}
    for i, k in enumerate(keys):
        out[assign[k]].append(i)
    return out


def symmetry_score(pil_img, size=128):
    """Frontal CXRs are near left-right symmetric; laterals are not."""
    a = np.asarray(pil_img.convert("L").resize((size, size)), np.float32)
    b = a[:, ::-1]
    a = (a - a.mean()) / (a.std() + 1e-8)
    b = (b - b.mean()) / (b.std() + 1e-8)
    return float((a * b).mean())


FRONTAL_THRESHOLD = 0.68

In [ ]:
class SpatialMLPAdapter(nn.Module):


    def __init__(self, vision_dim=CFG.VISION_DIM, llm_dim=2048,
                 raw_grid=CFG.RAW_GRID, pool_grid=CFG.POOL_GRID,
                 target_norm=1.0):
        super().__init__()
        self.raw_grid, self.pool_grid = raw_grid, pool_grid
        self.proj = nn.Linear(vision_dim, llm_dim)
        self.norm = nn.LayerNorm(llm_dim)
        self.scale = nn.Parameter(torch.tensor(target_norm / (llm_dim ** 0.5)))

    def forward(self, rad_out):
        x = rad_out[:, 1:, :]
        B, N, C = x.shape
        assert N == self.raw_grid ** 2, (
            f"expected {self.raw_grid**2} patches, got {N}")
        x = x.transpose(1, 2).reshape(B, C, self.raw_grid, self.raw_grid)
        x = F.adaptive_avg_pool2d(x, (self.pool_grid, self.pool_grid))
        x = x.flatten(2).transpose(1, 2)
        return self.norm(self.proj(x)) * self.scale


seed_everything()
print("adapter class ready")

adapter class ready


In [ ]:
def patient_level_split(manifest, frac=(0.80, 0.10, 0.10), seed=42):
    groups = {}
    for m in manifest:
        groups.setdefault(str(m["patient_id"]), []).append(m)
    abn = [k for k, v in groups.items() if any(x["abnormal"] for x in v)]
    nrm = [k for k, v in groups.items() if not any(x["abnormal"] for x in v)]
    rng = random.Random(seed); rng.shuffle(abn); rng.shuffle(nrm)
    out = {"train": [], "val": [], "test": []}
    ids = {"train": set(), "val": set(), "test": set()}
    for pool in (abn, nrm):
        a, b = int(len(pool)*frac[0]), int(len(pool)*(frac[0]+frac[1]))
        for nm, ks in [("train", pool[:a]), ("val", pool[a:b]), ("test", pool[b:])]:
            for k in ks:
                out[nm].extend(groups[k]); ids[nm].add(k)
    assert ids["train"].isdisjoint(ids["val"])
    assert ids["train"].isdisjoint(ids["test"])
    assert ids["val"].isdisjoint(ids["test"])
    return out["train"], out["val"], out["test"]


pc_train, pc_val, pc_test = patient_level_split(pc_all)
tr_abn = [m for m in pc_train if m["abnormal"]]
va_abn = [m for m in pc_val   if m["abnormal"]]
te_abn = [m for m in pc_test  if m["abnormal"]]

ns = sum(len(m["sentences"]) for m in tr_abn)
ng = sum(sum(1 for b in m["boxes"] if b) for m in tr_abn)
print(f"ABNORMAL-ONLY  train {len(tr_abn)} | val {len(va_abn)} | test {len(te_abn)}")
print(f"train sentences {ns} | with box {ng} ({100*ng/ns:.1f}%)")
print("\n(full split was 3643/456/456 at 68.0% abnormal)")

pc_train, pc_val, pc_test = patient_level_split(pc_all)
te_abn = [m for m in pc_test if m["abnormal"]]
te_nrm = [m for m in pc_test if not m["abnormal"]]
print(f"test: {len(te_abn)} abnormal | {len(te_nrm)} normal")

ABNORMAL-ONLY  train 2479 | val 310 | test 310
train sentences 6955 | with box 5034 (72.4%)

(full split was 3643/456/456 at 68.0% abnormal)
test: 310 abnormal | 146 normal


## 3. Load the model

In [ ]:
def build_model():
    from transformers import (AutoModel, AutoTokenizer, AutoModelForCausalLM,
                              BitsAndBytesConfig)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    print("Rad-DINO (frozen)...")
    vision = AutoModel.from_pretrained("microsoft/rad-dino").to(device).eval()
    for p in vision.parameters():
        p.requires_grad = False

    print("Qwen2.5-3B (4-bit) + LoRA...")
    tok = AutoTokenizer.from_pretrained(CFG.LLM_ID)
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                             bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16)
    base = AutoModelForCausalLM.from_pretrained(
        CFG.LLM_ID, quantization_config=bnb, device_map={"": 0})


    tok, base = add_coord_tokens(tok, base)
    base = prepare_model_for_kbit_training(base)


    lora = LoraConfig(r=CFG.LORA_R, lora_alpha=CFG.LORA_ALPHA,
                      lora_dropout=CFG.LORA_DROPOUT, bias="none",
                      task_type="CAUSAL_LM", target_modules=CFG.LORA_TARGETS)
    llm = get_peft_model(base, lora)
    llm.config.use_cache = False
    llm.gradient_checkpointing_enable()
    llm.enable_input_require_grads()


    n_new = len(coord_tokens())
    emb = llm.get_input_embeddings()
    emb.weight.requires_grad_(True)
    mask = torch.zeros(emb.weight.shape[0], 1,
                       device=emb.weight.device, dtype=emb.weight.dtype)
    mask[-n_new:] = 1.0
    emb.weight.register_hook(lambda g: g * mask)
    llm._n_new_tokens = n_new

    adapter = SpatialMLPAdapter(llm_dim=llm.config.hidden_size).to(device)

    n_ad = sum(p.numel() for p in adapter.parameters())
    n_lora = sum(p.numel() for n, p in llm.named_parameters()
                 if p.requires_grad and "embed_tokens" not in n)
    n_emb_eff = n_new * emb.weight.shape[1]
    n_frozen = sum(p.numel() for p in llm.parameters()) \
        + sum(p.numel() for p in vision.parameters())
    tot = n_ad + n_lora + n_emb_eff
    print(f"\nTRAINABLE PARAMETERS")
    print(f"  MLP adapter          {n_ad:>12,}")
    print(f"  LoRA (r={CFG.LORA_R})           {n_lora:>12,}")
    print(f"  coord embeddings     {n_emb_eff:>12,}  "
          f"({n_new} tokens x {emb.weight.shape[1]})")
    print(f"  ------------------------------------")
    print(f"  total trainable      {tot:>12,}  ({100*tot/max(1,n_frozen):.2f}% "
          f"of the frozen backbone)")
    return vision, tok, llm, adapter

In [ ]:
@torch.no_grad()
def encode_image(vision, px):
    return vision(px.to(device)).last_hidden_state


RUN, CKPT = "run_B_final_ep23", "ep2_final.pt"
from peft import set_peft_model_state_dict

vision, tok, llm, adapter = build_model()
st = torch.load(f"{OUT_ROOT}/{RUN}/{CKPT}", map_location=device, weights_only=False)
adapter.load_state_dict(st["adapter"])
set_peft_model_state_dict(llm, st["lora"])
with torch.no_grad():
    w = llm.get_input_embeddings().weight
    w[-68:] = st["coord"].to(w.dtype).to(w.device)
llm.eval(); adapter.eval()
print(f"loaded {RUN}/{CKPT} | loss {st.get('loss'):.4f} | "
      f"P(<box>) {st.get('p_box'):.3f}")

Rad-DINO (frozen)...


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

added 68 tokens | vocab 151733 | new-row spread 0.02339 (must be > 0)

TRAINABLE PARAMETERS
  MLP adapter             1,579,009
  LoRA (r=32)             59,867,136
  coord embeddings          139,264  (68 tokens x 2048)
  ------------------------------------
  total trainable        61,585,409  (3.34% of the frozen backbone)
loaded run_B_final_ep23/ep2_final.pt | loss 2.0581 | P(<box>) 0.960


## 4. Inference

Coordinate ordering is enforced during decoding. The model reliably chooses to
open a box (P = 0.96) but does not emit `<x><y><x><y>` in order — 33% of raw
boxes are corner-inverted. The constraint fixes ORDER only; coordinate
*selection* is the model's.

In [ ]:
# import torch, numpy as np

# X_IDS = torch.tensor(tok.convert_tokens_to_ids([f"<x{i}>" for i in range(32)]))
# Y_IDS = torch.tensor(tok.convert_tokens_to_ids([f"<y{i}>" for i in range(32)]))
# BOX_O = tok.convert_tokens_to_ids("<box>");  BOX_C = tok.convert_tokens_to_ids("</box>")
# OBJ_O = tok.convert_tokens_to_ids("<obj>");  OBJ_C = tok.convert_tokens_to_ids("</obj>")


# @torch.no_grad()
# def generate_constrained(study, max_new=200, temperature=0.7, top_p=0.9,
#                          max_obj=4):

#     px, _, _ = preprocess(Image.open(study["path"]))
#     vis = adapter(encode_image(vision, px[None]).float())
#     ids = tok(SYSTEM + INSTR_GROUNDREP + "<obj>", return_tensors="pt")["input_ids"].to(device)
#     emb = llm.get_input_embeddings()(ids)
#     inp = torch.cat([vis.to(emb.dtype), emb], dim=1)

#     out_ids, in_box, n_obj = [], -1, 1
#     for _ in range(max_new):
#         with torch.amp.autocast("cuda", dtype=torch.float16):
#             logits = llm(inputs_embeds=inp,
#                          attention_mask=torch.ones(inp.shape[:2], dtype=torch.long,
#                                                    device=device)).logits[0, -1].float()
#         if in_box >= 0:
#             allowed = (X_IDS if in_box % 2 == 0 else Y_IDS).to(device)
#             m = torch.full_like(logits, float("-inf")); m[allowed] = logits[allowed]
#             logits = m


#         else:
#             logits[BOX_C] = float("-inf")
#             # coordinates are ONLY legal inside a box — the model emits them in
#             # text position and they parse as garbage
#             logits[X_IDS.to(device)] = float("-inf")
#             logits[Y_IDS.to(device)] = float("-inf")
#             if out_ids and out_ids[-1] == OBJ_C:
#                 logits[OBJ_O] -= 3.0



#                     # discourage <obj> spirals

#         p = torch.softmax(logits / temperature, -1)
#         sp, si = p.sort(descending=True)
#         keep = (sp.cumsum(-1) - sp) < top_p
#         sp, si = sp[keep], si[keep]
#         nxt = si[torch.multinomial(sp / sp.sum(), 1)].item()

#         if in_box >= 0:
#             in_box += 1
#             if in_box == 4:
#                 out_ids.append(nxt); nxt = BOX_C; in_box = -1
#         elif nxt == BOX_O:
#             in_box = 0
#         elif nxt == OBJ_C:
#             n_obj += 1
#             out_ids.append(nxt)
#             break


#         out_ids.append(nxt)
#         if nxt == tok.eos_token_id or n_obj > max_obj:
#             break
#         e = llm.get_input_embeddings()(torch.tensor([[nxt]], device=device))
#         inp = torch.cat([inp, e.to(inp.dtype)], dim=1)

#     txt = tok.decode(out_ids, skip_special_tokens=False)
#     txt = txt.split("<|im_end|>")[0]
#     return "<obj>" + txt

In [ ]:
import torch, numpy as np

X_IDS = torch.tensor(tok.convert_tokens_to_ids([f"<x{i}>" for i in range(32)]))
Y_IDS = torch.tensor(tok.convert_tokens_to_ids([f"<y{i}>" for i in range(32)]))
BOX_O = tok.convert_tokens_to_ids("<box>");  BOX_C = tok.convert_tokens_to_ids("</box>")
OBJ_O = tok.convert_tokens_to_ids("<obj>");  OBJ_C = tok.convert_tokens_to_ids("</obj>")


@torch.no_grad()
def generate_constrained(study, max_new=200, temperature=0.7, top_p=0.9,
                         max_obj=4):

    px, _, _ = preprocess(Image.open(study["path"]))
    vis = adapter(encode_image(vision, px[None]).float())
    ids = tok(SYSTEM + INSTR_GROUNDREP + "<obj>", return_tensors="pt")["input_ids"].to(device)
    emb = llm.get_input_embeddings()(ids)
    inp = torch.cat([vis.to(emb.dtype), emb], dim=1)

    out_ids, in_box, n_obj = [], -1, 1
    for _ in range(max_new):
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = llm(inputs_embeds=inp,
                         attention_mask=torch.ones(inp.shape[:2], dtype=torch.long,
                                                   device=device)).logits[0, -1].float()
        if in_box >= 0:
            allowed = (X_IDS if in_box % 2 == 0 else Y_IDS).to(device)
            m = torch.full_like(logits, float("-inf")); m[allowed] = logits[allowed]
            logits = m


        else:
            logits[BOX_C] = float("-inf")
            # coordinates are ONLY legal inside a box — the model emits them in
            # text position and they parse as garbage
            logits[X_IDS.to(device)] = float("-inf")
            logits[Y_IDS.to(device)] = float("-inf")
            if out_ids and out_ids[-1] == OBJ_C:
                logits[OBJ_O] -= 3.0



                    # discourage <obj> spirals

        p = torch.softmax(logits / temperature, -1)
        sp, si = p.sort(descending=True)
        keep = (sp.cumsum(-1) - sp) < top_p
        sp, si = sp[keep], si[keep]
        nxt = si[torch.multinomial(sp / sp.sum(), 1)].item()

        if in_box >= 0:
            in_box += 1
            if in_box == 4:
                out_ids.append(nxt); nxt = BOX_C; in_box = -1
        elif nxt == BOX_O:
            in_box = 0
        elif nxt == OBJ_C:
            n_obj += 1
            out_ids.append(nxt)
            break                      # one finding per generation


        out_ids.append(nxt)
        if nxt == tok.eos_token_id or n_obj > max_obj:
            break
        e = llm.get_input_embeddings()(torch.tensor([[nxt]], device=device))
        inp = torch.cat([inp, e.to(inp.dtype)], dim=1)

    txt = tok.decode(out_ids, skip_special_tokens=False)
    txt = txt.split("<|im_end|>")[0]
    return "<obj>" + txt

In [ ]:
def normalise_box(b):
    """Canonicalise to x0<x1, y0<y1 — 33% of raw boxes are corner-inverted."""
    return (min(b[0], b[2]), min(b[1], b[3]), max(b[0], b[2]), max(b[1], b[3]))


def load_any_cxr(path, img_size=518):

    a = np.array(Image.open(path)).astype(np.float32)
    if a.ndim == 3:
        a = a[..., 0]
    a = (a - a.min()) / max(1e-6, a.max() - a.min()) * 255.0
    im = Image.fromarray(a.astype(np.uint8)).convert("RGB")
    W, H = im.size
    sc = img_size / min(W, H)
    return im.resize((int(W*sc), int(H*sc)), Image.LANCZOS)


def infer(path, temperature=0.7):

    raw = generate_constrained({"path": path}, max_new=200,
                               temperature=temperature)
    return [(t.strip(), [normalise_box(b) for b in bl])
            for t, bl in parse_generation(raw) if t.strip()]


# sanity check against the evaluation numbers
n_box = 0
for s in te_abn[:8]:
    out = infer(s["path"])
    nb = sum(len(b) for _, b in out)
    n_box += nb > 0
    print(f"  [{nb} box] {out[0][0][:64] if out else '(none)'}")
print(f"\n{n_box}/8 produced boxes  (evaluation gave 37/50 = 74%)")

  [0 box] No significant findings.
  [1 box] Perihilar and hilar bronchial thickening.
  [1 box] Small aortic elongation.
  [2 box] Bilateral pulmonary infiltration.
  [0 box] No infiltrates.
  [1 box] Aortic elongation.
  [1 box] Chronic changes in the right lung parenchyma.
  [0 box] Chronic changes in the pulmonary parenchyma.

5/8 produced boxes  (evaluation gave 37/50 = 74%)


## 5. Gradio interface

Boxes are drawn only for findings the model chose to localise. When nothing is
localisable the image is returned unannotated with an explicit statement, rather
than a blank result that could be mistaken for a failure.

In [ ]:
import gradio as gr, traceback, time
import numpy as np
from PIL import Image, ImageDraw, ImageFont

PALETTE = [(230, 25, 75), (60, 180, 75), (67, 99, 216), (245, 130, 49),
           (145, 30, 180), (66, 212, 244)]


def draw_boxes(im, findings, lw=3):
    im = im.convert("RGB").copy()
    d = ImageDraw.Draw(im)
    W, H = im.size

    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
    except Exception:
        font = ImageFont.load_default()

    n = 0
    for i, (_, boxes) in enumerate(findings):
        col = PALETTE[i % len(PALETTE)]

        for b in boxes:
            x0, y0, x1, y1 = b[0]*W, b[1]*H, b[2]*W, b[3]*H

            d.rectangle([x0, y0, x1, y1], outline=col, width=lw)
            d.rectangle([x0, max(0, y0-26), x0+26, y0], fill=col)
            d.text(
                (x0+7, max(0, y0-25)),
                str(i+1),
                fill=(255, 255, 255),
                font=font
            )
            n += 1

    return im, n


def analyse(image):
    """Wrapped so failures show in the UI instead of Gradio's bare 'Error'."""
    try:
        if image is None:
            return None, "Upload a chest radiograph to begin.", ""

        tmp = "/content/_ui_input.png"

        if isinstance(image, np.ndarray):
            Image.fromarray(image).save(tmp)
        else:
            image.save(tmp)

        im = load_any_cxr(tmp)
        im.save(tmp)

        t0 = time.time()

        # No temperature parameter
        findings = infer(tmp)

        dt = time.time() - t0

        annotated, n_boxes = draw_boxes(im, findings)

        grounded = [(t, b) for t, b in findings if b]
        ungrounded = [(t, b) for t, b in findings if not b]

        lines = []

        if not findings:
            lines.append("**No findings generated.**\n")

        else:
            if grounded:
                lines.append("### Localised findings\n")

                for i, (t, b) in enumerate(findings):
                    if not b:
                        continue

                    lines.append(f"**{i+1}.** {t}")

                    for x in b:
                        lines.append(
                            f"&nbsp;&nbsp;&nbsp;&nbsp;"
                            f"`({x[0]:.2f}, {x[1]:.2f}) → "
                            f"({x[2]:.2f}, {x[3]:.2f})`"
                        )

                lines.append("")

            if ungrounded:
                lines.append("### Findings without localisation\n")

                for t, _ in ungrounded:
                    lines.append(f"- {t}")

                lines.append("")

            if not grounded:
                lines.append(
                    "---\n"
                    "**Nothing to indicate on the radiograph.** "
                    "The model reported no localisable finding. "
                    "Negations, statements of normality and "
                    "non-localisable findings are reported without a "
                    "box by design.\n"
                )

        meta = (
            f"`{len(findings)} findings · {n_boxes} boxes · {dt:.1f}s`\n\n"
            f"Regional localisation only (mean IoU 0.185). Research prototype — "
            f"**not for clinical use**."
        )

        return annotated, "\n".join(lines), meta

    except Exception:
        tb = traceback.format_exc()
        print(tb)

        return None, f"### Error\n```\n{tb[-1800:]}\n```", ""


examples = [[te_abn[i]["path"]] for i in range(6)] + \
           [[te_nrm[i]["path"]] for i in range(2)]


with gr.Blocks(
    title="Grounded Radiology Reporting",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        "# Grounded Radiology Report Generation\n"
        "Rad-DINO → linear adapter → Qwen2.5-3B (LoRA) with coordinate "
        "tokens. Trained on PadChest-GR."
    )

    with gr.Row():

        with gr.Column(scale=1):

            inp = gr.Image(
                type="pil",
                label="Chest radiograph",
                height=430
            )

            btn = gr.Button(
                "Generate report",
                variant="primary"
            )

            gr.Examples(
                examples=examples,
                inputs=inp,
                label="Test-set examples"
            )

        with gr.Column(scale=1):

            outimg = gr.Image(
                label="Findings",
                height=430
            )

            outtxt = gr.Markdown()
            outmeta = gr.Markdown()

    btn.click(
        analyse,
        [inp],
        [outimg, outtxt, outmeta]
    )


demo.launch(
    share=True,
    debug=True,
    show_error=True
)

/tmp/ipykernel_3672/3008277058.py:129: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://299c21ad67ca78cde2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
